# In-context Learning: Zero-shot, Few-shot & CoT (5-fold CV)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DSPagan/llms-time-complexity/blob/main/notebooks/in_context_learning.ipynb)

Evaluate the **base** `Llama 3.1 8B Instruct` model (no fine-tuning) with **5-fold cross-validation**, so every metric comes with a **mean ± std** across folds. Prompts: two zero-shot (`direct`, `persona`), few-shot, and chain-of-thought. Metrics come from `src/evaluate.py`.

In [ ]:
# Clone the repo to get the code (src/), the data, and the pinned requirements.
!git clone https://github.com/DSPagan/llms-time-complexity.git
%cd llms-time-complexity

In [ ]:
# Install the exact pinned versions so every experiment runs on the same stack.
# Installing "latest" instead lets Unsloth drift between runs (and 2026.7.2 fails to
# resolve the 4-bit model repo), which would make the results incomparable.
!pip install --no-cache-dir -r requirements-lock.txt

In [ ]:
import os, sys, json, random
import numpy as np
from collections import defaultdict

sys.path.insert(0, os.getcwd())

from unsloth import FastLanguageModel
from src.load_model import load_model
from src.prepare_data import load_clean, stratified_folds
from src.evaluate import evaluate, plot_confusion_matrix
from src.cache import ResponseCache

In [ ]:
max_seq_length = 8192  # few-shot packs several examples per prompt
model, tokenizer = load_model(max_seq_length=max_seq_length)
FastLanguageModel.for_inference(model)

K = 5
folds = stratified_folds(load_clean(), k=K, seed=42)
print("fold sizes:", [len(f) for f in folds])

# Disk-backed response cache so an interrupted run can resume. Upload a saved
# outputs/response_cache.json to continue where a previous session left off.
cache = ResponseCache()
print("cached responses:", len(cache))

# Set LIMIT to a small number (e.g. 10) to smoke-test; it caps test items per fold.
LIMIT = None

def generate(prompt, max_new_tokens=64):
    key = f"{max_new_tokens}\n{prompt}"
    hit = cache.get(key)
    if hit is not None:
        return hit
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    if inputs.shape[1] > max_seq_length:
        return None  # too long: not cached (cheap to recheck)
    out = model.generate(
        input_ids=inputs, do_sample=False, max_new_tokens=max_new_tokens,
        use_cache=True, no_repeat_ngram_size=4,
    )
    result = tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()
    cache.set(key, result)
    return result

In [ ]:
# --- Zero-shot prompts ---
# Single source of truth for the label set, presented identically in every prompt.
OPTIONS = "O(1), O(logn), O(n), O(nlogn), O(n^2), O(n^3), exponential"

def prompt_direct(code):
    return (
        "Analyze the time complexity of the following code.\n"
        f"Choose exactly one of the following options: {OPTIONS}.\n"
        "Answer with only the chosen option, without any explanation.\n"
        "Code:\n"
        f"{code}"
    )

def prompt_persona(code):
    return (
        "You are an expert in algorithm analysis and time complexity.\n"
        "Analyze the time complexity of the following code.\n"
        f"Choose exactly one of the following options: {OPTIONS}.\n"
        "Answer with only the chosen option, without any explanation.\n"
        "Code:\n"
        f"{code}"
    )

# --- Few-shot: one example per class, drawn from each fold's own training split ---

FEWSHOT_CLASSES = [
    ("constant", "O(1)"), ("logn", "O(logn)"), ("linear", "O(n)"),
    ("nlogn", "O(nlogn)"), ("quadratic", "O(n^2)"), ("cubic", "O(n^3)"),
    ("exponential", "exponential"),
]

def pick_fewshot_examples(train, seed=42):
    by_class = defaultdict(list)
    for item in train:
        by_class[item["complexity"]].append(item["src"])
    rng = random.Random(seed)
    return {cls: rng.choice(by_class[cls]) for cls, _ in FEWSHOT_CLASSES}

def make_few_shot_prompt(examples):
    def prompt_few_shot(code):
        parts = [
            "Analyze the time complexity of the following code.",
            f"Choose exactly one of the following options: {OPTIONS}.",
            "Answer with only the chosen option, without any explanation.",
            "Here are some examples:",
            "",
        ]
        for i, (cls, disp) in enumerate(FEWSHOT_CLASSES, start=1):
            parts += [f"Example {i}:", "Code:", examples[cls], f"Complexity: {disp}", ""]
        parts += ["Now analyze this code:", "Code:", code, "Complexity:"]
        return "\n".join(parts)
    return prompt_few_shot

# --- Chain-of-thought prompt (beyond the thesis: reason first, then answer) ---

def prompt_cot(code):
    return (
        "You are analyzing the worst-case time complexity of a Python program.\n\n"
        "Reason step by step:\n"
        "1. Identify the loops and recursion, and how many times each runs in terms "
        "of the input size n.\n"
        "2. Combine them to find the dominant term.\n"
        "3. Map that term to the closest complexity class.\n\n"
        f"Choose exactly one of: {OPTIONS}.\n\n"
        "After your reasoning, end with a single line exactly of the form:\n"
        "Final complexity: <chosen option>\n\n"
        "Code:\n"
        f"{code}"
    )

In [ ]:
results = {}

def extract_final(text):
    # Chain-of-thought: keep only what follows the "Final complexity:" marker.
    if text and "final complexity:" in text.lower():
        return text.lower().rsplit("final complexity:", 1)[-1]
    return text

def run_cv(prompt_for_fold, max_new_tokens=64, post=None):
    """Evaluate one experiment across the K folds.
    prompt_for_fold(fold_index, train) returns the prompt builder for that fold."""
    accs, f1s, cm_sum = [], [], None
    for i in range(K):
        test = folds[i] if LIMIT is None else folds[i][:LIMIT]
        train = [x for j in range(K) if j != i for x in folds[j]]
        build = prompt_for_fold(i, train)
        raw = [generate(build(item["src"]), max_new_tokens) for item in test]
        cache.save()  # persist responses after each fold so a crash can resume
        if post:
            raw = [post(r) for r in raw]
        res = evaluate([item["complexity"] for item in test], raw)
        accs.append(res["accuracy"]); f1s.append(res["macro_f1"])
        cm_sum = res["confusion_matrix"] if cm_sum is None else cm_sum + res["confusion_matrix"]
        print(f"  fold {i}: accuracy={res['accuracy']:.3f}  macro_f1={res['macro_f1']:.3f}  "
              f"unclassified={res['n_unclassified']}")
    return accs, f1s, cm_sum

def report(name, accs, f1s, cm, save_path):
    print(f"{name}: accuracy {np.mean(accs):.3f} +/- {np.std(accs, ddof=1):.3f}   "
          f"macro_f1 {np.mean(f1s):.3f} +/- {np.std(f1s, ddof=1):.3f}\n")
    results[name] = {"acc_mean": float(np.mean(accs)), "acc_std": float(np.std(accs, ddof=1)),
                     "f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s, ddof=1))}
    plot_confusion_matrix(cm, title=f"{name} (5-fold CV)", save_path=save_path)

## Zero-shot experiments

Two prompts (`direct` and `persona`), each evaluated across the 5 folds.

In [ ]:
print("zero-shot direct")
accs, f1s, cm = run_cv(lambda i, train: prompt_direct)
report("zero-shot direct", accs, f1s, cm, "figures/CM_zeroshot_direct.png")

print("zero-shot persona")
accs, f1s, cm = run_cv(lambda i, train: prompt_persona)
report("zero-shot persona", accs, f1s, cm, "figures/CM_zeroshot_persona.png")

## Few-shot experiment

One example per class, drawn from **each fold's own training split** (no leakage). Cross-validation captures how much the metric varies across folds.

In [ ]:
print("few-shot")
accs, f1s, cm = run_cv(
    lambda i, train: make_few_shot_prompt(pick_fewshot_examples(train, seed=42))
)
report("few-shot", accs, f1s, cm, "figures/CM_fewshot.png")

## Chain-of-thought (beyond the thesis)

A better-structured prompt that reasons step by step before answering, ending with a parseable `Final complexity:` line. Complexity estimation is a reasoning task, so this usually helps zero-shot the most.

In [ ]:
print("chain-of-thought")
accs, f1s, cm = run_cv(lambda i, train: prompt_cot, max_new_tokens=512, post=extract_final)
report("chain-of-thought", accs, f1s, cm, "figures/CM_cot.png")

## Results summary (5-fold CV)

In [ ]:
print(f"{'Experiment':<20}  {'Accuracy':<16}  {'Macro F1':<16}")
for name, r in results.items():
    print(f"{name:<20}  {r['acc_mean']:.3f} +/- {r['acc_std']:.3f}   "
          f"{r['f1_mean']:.3f} +/- {r['f1_std']:.3f}")